# Clase 14 — Proyecto integrador

**Curso:** Estadística Aplicada con Python · **Bloque 5:** Cierre
**Tiempo estimado:** 120 minutos · **Capítulo del manual:** `manual/14_proyecto_integrador.md`

## Qué vas a ver en esta clase

Una **demostración completa** del flujo de un análisis, de la pregunta al informe, con los pingüinos de Palmer (no es uno de los tres casos del proyecto: así puedes seguir el método sin ver resuelto tu propio caso).

1. **Pregunta de negocio → pregunta estadística → decisión.**
2. **Auditoría** de los datos (tipos, ausentes, duplicados, rangos) y **exploración** antes de modelar.
3. **Elegir y justificar el método** (ANOVA/Tukey o Kruskal-Wallis; regresión con una dummy) y **comprobar los supuestos**.
4. **Resultados con incertidumbre:** efecto + intervalo de confianza, no solo el p-value.
5. Un **experimento guiado**: la paradoja de Simpson en datos reales.
6. **Límites** y un **informe de una página** escrito en una celda.

## Requisitos

- Todo el curso: clases 1-13. Cada paso remite a la clase donde se explicó la herramienta; aquí **no** se vuelve a explicar la teoría.
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen en Google Colab).

> **Datos:** el notebook carga `datos/penguins.csv` (Palmer Penguins: Horst, Hill y Gorman, 2020). Si no lo encuentra, lo descarga con `seaborn`; y si tampoco puede, genera un dataset **sintético** con las mismas columnas (los números cambiarán). Después de la clase, el notebook de práctica usa los tres casos del proyecto.

---
## Celda 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from IPython.display import Markdown, display

print("Librerías cargadas correctamente.")

---
## 1. La ruta de un análisis: de la pregunta a la decisión

Un análisis que empieza con `import pandas` ya empieza mal. Empieza con una **decisión**: alguien tiene que hacer (o dejar de hacer) algo según lo que descubras. Nuestra demostración:

| Paso | En nuestro caso |
|---|---|
| **Pregunta de negocio** | Una estación de investigación quiere identificar la especie de un pingüino con medidas rápidas de campo. ¿Distinguen las especies por su masa corporal? |
| **Pregunta estadística** | ¿La masa media difiere entre Adelie, Chinstrap y Gentoo? ¿Cuánto, con qué incertidumbre? |
| **Decisión que informa** | Si la masa separa bien dos especies pero no las otras dos, la balanza sirve solo para parte de la identificación. |
| **Herramientas** | Auditoría (clases 2-3), ANOVA y post-hoc (clase 8), bootstrap (clase 6), regresión (clase 10), paradoja de Simpson (clase 13). |

> **Regla de la clase:** antes de cada celda de código escribe qué esperas ver. Al final de cada bloque comprueba si acertaste.

---
## 2. Auditoría de los datos

Antes de calcular nada, **conoce lo que tienes**: cuántas filas, qué tipos, qué falta, qué se repite y si los rangos tienen sentido (clases 2 y 3). Definimos el cargador (copia local → `seaborn` → sintético) y hacemos la auditoría.

In [ ]:
def generar_penguins_sintetico(semilla=2026):
    """Dataset SINTÉTICO con las mismas columnas que `penguins` (ejemplo ilustrativo, NO son pingüinos reales)."""
    r = np.random.default_rng(semilla)
    especies = {
        "Adelie":    dict(n=152, pico=(38.8, 2.7), prof=(18.3, 1.2), aleta=(190, 6.5), masa=(3700, 460), islas=["Torgersen", "Biscoe", "Dream"]),
        "Chinstrap": dict(n=68,  pico=(48.8, 3.3), prof=(18.4, 1.1), aleta=(196, 7.1), masa=(3733, 384), islas=["Dream"]),
        "Gentoo":    dict(n=124, pico=(47.5, 3.1), prof=(15.0, 1.0), aleta=(217, 6.5), masa=(5076, 504), islas=["Biscoe"]),
    }
    bloques = []
    for esp, c in especies.items():
        n = c["n"]
        sexo = r.choice(["Male", "Female"], size=n)
        efecto = np.where(sexo == "Male", 0.5, -0.5)
        def medida(par):
            return par[0] + efecto * par[1] + r.normal(0, par[1] * 0.8, size=n)
        bloques.append(pd.DataFrame({
            "species": esp, "island": r.choice(c["islas"], size=n),
            "bill_length_mm": np.round(medida(c["pico"]), 1), "bill_depth_mm": np.round(medida(c["prof"]), 1),
            "flipper_length_mm": np.round(medida(c["aleta"])), "body_mass_g": np.round(medida(c["masa"]) / 25) * 25,
            "sex": sexo}))
    d = pd.concat(bloques, ignore_index=True)
    idx = r.choice(len(d), size=11, replace=False)
    d.loc[idx[:2], ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]] = np.nan
    d.loc[idx[2:], "sex"] = np.nan
    return d


def cargar_penguins():
    """Devuelve (DataFrame, origen): CSV local, si no descarga de seaborn, si no datos sintéticos."""
    for ruta in ("../datos/penguins.csv", "datos/penguins.csv", "penguins.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV local"
    try:
        import seaborn as sns
        return sns.load_dataset("penguins"), "seaborn"
    except Exception:
        return generar_penguins_sintetico(), "sintético"


pen, origen = cargar_penguins()
print("Origen de los datos:", origen)
print("Filas y columnas:", pen.shape)
print()
print(pen.dtypes.to_string())

In [ ]:
auditoria = pd.DataFrame({
    "ausentes": pen.isna().sum(),
    "% ausentes": (pen.isna().mean() * 100).round(1),
    "valores distintos": pen.nunique(),
})
print(auditoria.to_string())
print()
print("Filas duplicadas:", int(pen.duplicated().sum()))
print()
print(pen.describe().round(1).T[["count", "min", "50%", "max"]].to_string())
print()
print(pen["species"].value_counts().to_string())

### ¿Qué observas?

- Hay **344** filas; faltan **2** en cada medida física (la misma pareja de filas) y **11** en `sex` (incluidas esas dos). No hay duplicados.
- Los ausentes no son un detalle: la regla es **decidir** qué hacer con ellos y **decirlo**. Aquí no imputaremos; cada análisis usará las filas completas para las variables que necesita (`n` = 342 en la masa).
- Los rangos son plausibles (masas de 2 700 a 6 300 g, aletas de 172 a 231 mm): ningún valor imposible. Si hubiera aparecido una masa de 0 o de 60 000, el análisis habría empezado por ahí.
- Las especies están **desequilibradas** (152, 124 y 68): no es un problema para el ANOVA, pero conviene recordarlo al leer los intervalos.

In [ ]:
print("Filas con algún ausente:")
print(pen[pen.isna().any(axis=1)].groupby("species").size().to_string())

def completos(df, columnas):
    """Filas completas para las columnas de un análisis; dice cuántas se descartan."""
    sub = df.dropna(subset=columnas)
    print(f"Análisis con {', '.join(columnas)}: n = {len(sub)} (se descartan {len(df) - len(sub)})")
    return sub

masa = completos(pen, ["species", "body_mass_g"])

---
## 3. Explorar antes de modelar

Los gráficos y los resúmenes por grupo dicen qué modelo tiene sentido **antes** de ajustarlo (clases 2 y 3). Aquí queremos ver la forma y la dispersión de la masa en cada especie.

In [ ]:
resumen = masa.groupby("species")["body_mass_g"].agg(["count", "mean", "std", "median"]).round(1)
resumen.columns = ["n", "media", "desv. típica", "mediana"]
print(resumen.to_string())

especies = ["Adelie", "Chinstrap", "Gentoo"]
colores = {"Adelie": "tab:orange", "Chinstrap": "tab:purple", "Gentoo": "tab:green"}
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].boxplot([masa.loc[masa["species"] == e, "body_mass_g"] for e in especies], labels=especies)
ejes[0].set_title("Masa corporal por especie"); ejes[0].set_ylabel("gramos")
for e in especies:
    ejes[1].hist(masa.loc[masa["species"] == e, "body_mass_g"], bins=15, alpha=0.55, color=colores[e], label=e)
ejes[1].set_title("Distribución de la masa"); ejes[1].set_xlabel("gramos"); ejes[1].legend()
plt.tight_layout(); plt.show()

### ¿Qué observas?

- **Gentoo** está claramente por encima (media ≈ 5 076 g); **Adelie** y **Chinstrap** casi coinciden (≈ 3 701 y 3 733 g): sus cajas se solapan casi por completo.
- La dispersión no es idéntica (desviación típica ≈ 459, 384 y 504 g): un aviso para el supuesto de varianzas iguales.
- Las distribuciones son razonablemente simétricas, sin valores atípicos extremos.
- Ya podemos **anticipar** la respuesta: se separa bien Gentoo del resto; entre Adelie y Chinstrap la masa no ayuda. El análisis formal debe **cuantificarlo con su incertidumbre**, no descubrirlo.

---
## 4. Elegir y justificar el método

La elección **no** es una costumbre: se deduce de la pregunta y de la naturaleza de los datos. La tabla de decisión que usarás en el proyecto (detalle en el capítulo, sección 4):

| Pregunta | Datos | Método | Clase |
|---|---|---|---|
| Media de un grupo, con incertidumbre | Numérica | IC t o bootstrap | 6 |
| Comparar 2 grupos numéricos | Numérica + 2 grupos | Welch / Mann-Whitney | 7 |
| Comparar 3 o más grupos numéricos | Numérica + ≥ 3 grupos | **ANOVA + Tukey** / Kruskal-Wallis | 8 |
| Asociación entre categorías | Dos categóricas | Chi-cuadrado / Fisher | 9 |
| Muchos contrastes a la vez | Varios p-values | Bonferroni / Holm / BH | 9 |
| Relación numérica, "a igualdad de lo demás" | Respuesta numérica | **Regresión lineal** (+ HC3, VIF) | 10-12 |
| Respuesta sí/no | Respuesta binaria | Regresión logística | 11 |
| Comparar una propuesta con el control | Experimento aleatorizado | Test A/B (+ SRM) | 13 |

Nuestra pregunta (masa de **tres** especies) apunta al **ANOVA**. Antes de fiarnos de él, comprobamos sus supuestos (clase 8): normalidad de los residuos y homogeneidad de varianzas.

In [ ]:
modelo_masa = smf.ols("body_mass_g ~ C(species)", data=masa).fit()
grupos = [masa.loc[masa["species"] == e, "body_mass_g"].values for e in especies]

print("Shapiro-Wilk de los residuos: p =", round(stats.shapiro(modelo_masa.resid)[1], 3))
print("Levene (varianzas iguales): p =", round(stats.levene(*grupos)[1], 4))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(modelo_masa.resid, line="s", ax=ejes[0]); ejes[0].set_title("QQ de los residuos")
ejes[1].scatter(modelo_masa.fittedvalues, modelo_masa.resid, s=10, alpha=0.5)
ejes[1].axhline(0, color="crimson"); ejes[1].set_xlabel("valores ajustados"); ejes[1].set_ylabel("residuos")
ejes[1].set_title("Residuos frente a ajustados")
plt.tight_layout(); plt.show()

### ¿Qué observas?

- **Normalidad:** el QQ sigue la diagonal y Shapiro da p ≈ 0,05: en el límite, sin desviaciones graves. Con 342 datos, el teorema del límite central (clase 5) hace el resto.
- **Varianzas:** Levene rechaza la igualdad (p ≈ 0,006). La diferencia entre las desviaciones (384 a 504 g) es moderada, pero el supuesto **no se cumple estrictamente**.
- **Decisión de método:** usamos el **ANOVA de Welch** (no exige varianzas iguales) y, como comprobación, **Kruskal-Wallis** (no exige normalidad). Si ambos coinciden, la conclusión no depende del supuesto dudoso. Para los pares, **Tukey** (con la cautela de que asume varianzas parecidas) más un **bootstrap** como segunda opinión.

In [ ]:
welch = anova_oneway(grupos, use_var="unequal", welch_correction=True)
kw = stats.kruskal(*grupos)
eta2 = modelo_masa.rsquared                       # proporción de la variación de la masa explicada por la especie
print(f"ANOVA Welch: F = {welch.statistic:.1f}, p = {welch.pvalue:.2g}")
print(f"Kruskal-Wallis: H = {kw[0]:.1f}, p = {kw[1]:.2g}")
print(f"Tamaño del efecto: eta² = {eta2:.3f}")

tukey = pairwise_tukeyhsd(masa["body_mass_g"], masa["species"], alpha=0.05)
tabla_tukey = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
print()
print(tabla_tukey.to_string(index=False))

### ¿Qué observas?

- Welch y Kruskal-Wallis coinciden con contundencia (p prácticamente 0): la masa **sí** difiere entre especies. El tamaño del efecto es **muy grande**: la especie explica el 67 % de la variación de la masa.
- Pero "difiere" no dice **quién**. Tukey responde con las diferencias por pares y su IC: Gentoo pesa ≈ 1 375 g más que Adelie y ≈ 1 343 g más que Chinstrap, con IC lejos de 0. **Adelie y Chinstrap no se distinguen** (≈ +32 g, IC de −127 a +191 g, que incluye el 0).
- Un p-value pequeño sin este desglose habría dado una conclusión engañosa: "las especies difieren en masa" no es lo mismo que "las tres son distinguibles".

In [ ]:
# Segunda opinión: bootstrap percentil de la diferencia de medias Gentoo − Adelie (clase 6)
g_gen = masa.loc[masa["species"] == "Gentoo", "body_mass_g"].values
g_ade = masa.loc[masa["species"] == "Adelie", "body_mass_g"].values
difs = np.array([rng.choice(g_gen, g_gen.size).mean() - rng.choice(g_ade, g_ade.size).mean() for _ in range(5000)])
boot_ic = np.percentile(difs, [2.5, 97.5])
fila = tabla_tukey[(tabla_tukey["group1"] == "Adelie") & (tabla_tukey["group2"] == "Gentoo")].iloc[0]
print(f"Diferencia Gentoo − Adelie: {g_gen.mean() - g_ade.mean():.0f} g")
print(f"IC 95 % bootstrap: [{boot_ic[0]:.0f}, {boot_ic[1]:.0f}]   IC 95 % Tukey: [{fila['lower']:.0f}, {fila['upper']:.0f}]")

fig, ax = plt.subplots(figsize=(7, 2.8))
for i, r_ in tabla_tukey.iterrows():
    ax.errorbar(r_["meandiff"], i, xerr=[[r_["meandiff"] - r_["lower"]], [r_["upper"] - r_["meandiff"]]], fmt="o", color="steelblue", capsize=4)
ax.axvline(0, color="crimson", linestyle="--")
ax.set_yticks(range(len(tabla_tukey))); ax.set_yticklabels(tabla_tukey["group1"] + " → " + tabla_tukey["group2"])
ax.set_xlabel("diferencia de medias (g) con IC 95 %"); ax.set_title("Comparaciones por pares (Tukey)")
plt.tight_layout(); plt.show()

### ¿Qué observas?

- El IC bootstrap y el de Tukey son casi idénticos: la conclusión **no depende** de un método concreto. Ese es el valor de una segunda opinión.
- El gráfico resume la historia mejor que un p-value: dos intervalos lejos del 0 (los que implican a Gentoo) y uno que lo cruza (Adelie-Chinstrap).

---
## 5. Regresión con la especie como dummy

Segunda pregunta: ¿cuánto **más larga** es la aleta cuando el pingüino pesa más, **a igualdad de especie**? Es una regresión múltiple con una variable categórica (clase 10) y errores robustos HC3 por prudencia. Comprobamos también la multicolinealidad con el VIF (clase 12).

In [ ]:
aleta = completos(pen, ["species", "body_mass_g", "flipper_length_mm"])
m_simple = smf.ols("flipper_length_mm ~ body_mass_g", data=aleta).fit()
m_mult = smf.ols("flipper_length_mm ~ body_mass_g + C(species)", data=aleta).fit()
m_hc3 = smf.ols("flipper_length_mm ~ body_mass_g + C(species)", data=aleta).fit(cov_type="HC3")

print(f"Sin la especie: +{m_simple.params['body_mass_g'] * 100:.2f} mm de aleta por cada 100 g")
print(f"Con la especie: +{m_hc3.params['body_mass_g'] * 100:.2f} mm por cada 100 g (IC 95 % HC3: {m_hc3.conf_int().loc['body_mass_g', 0] * 100:.2f} a {m_hc3.conf_int().loc['body_mass_g', 1] * 100:.2f})")
print(f"Gentoo frente a Adelie, a igual masa: +{m_hc3.params['C(species)[T.Gentoo]']:.1f} mm")
print(f"R² corregido: {m_mult.rsquared_adj:.3f}")

X = sm.add_constant(pd.get_dummies(aleta[["body_mass_g", "species"]], drop_first=True).astype(float))
vifs = {col: round(variance_inflation_factor(X.values, i), 2) for i, col in enumerate(X.columns) if col != "const"}
print("VIF:", vifs)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for e in especies:
    s = aleta[aleta["species"] == e]
    ejes[0].scatter(s["body_mass_g"], s["flipper_length_mm"], s=12, alpha=0.6, color=colores[e], label=e)
ejes[0].set_xlabel("masa (g)"); ejes[0].set_ylabel("aleta (mm)"); ejes[0].legend(); ejes[0].set_title("Aleta y masa por especie")
ejes[1].scatter(m_mult.fittedvalues, m_mult.resid, s=10, alpha=0.5); ejes[1].axhline(0, color="crimson")
ejes[1].set_xlabel("valores ajustados"); ejes[1].set_ylabel("residuos"); ejes[1].set_title("Diagnóstico de los residuos")
plt.tight_layout(); plt.show()

### ¿Qué observas?

- **Sin controlar la especie** el coeficiente casi se **duplica** (+1,53 frente a +0,84 mm por 100 g): parte de la relación aparente es simplemente que Gentoo es más pesado **y** tiene las aletas más largas. Es el mismo mecanismo de confusión de la clase 10.
- Los VIF son bajos (entre 1,2 y 3,2, todos por debajo de 5): la masa y las especies no son redundantes para este modelo.
- Los residuos no muestran curvatura ni un abanico claro: el modelo lineal es razonable. El IC con HC3 es la cifra que comunicamos.
- Recuerda: esto es una asociación en datos **observacionales**. No decimos que engordar "alargue" la aleta de un pingüino.

---
## 6. Experimento guiado: la paradoja de Simpson

La lección 13 mostró que una tendencia puede **invertirse** al mezclar grupos. ¿Ocurre con datos reales? Miramos la correlación entre la **longitud** y la **profundidad** del pico.

**Antes de ejecutar, predice:** si dibujas todos los pingüinos juntos, ¿la correlación entre longitud y profundidad del pico será positiva o negativa? ¿Y dentro de cada especie?

In [ ]:
pico = completos(pen, ["species", "bill_length_mm", "bill_depth_mm"])
r_total = pico["bill_length_mm"].corr(pico["bill_depth_mm"])
r_esp = {e: pico.loc[pico["species"] == e, "bill_length_mm"].corr(pico.loc[pico["species"] == e, "bill_depth_mm"]) for e in especies}
print(f"Correlación en el total: r = {r_total:.3f}")
for e, r_ in r_esp.items():
    print(f"Dentro de {e}: r = {r_:.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
ejes[0].scatter(pico["bill_length_mm"], pico["bill_depth_mm"], s=12, alpha=0.6, color="gray")
b1, b0 = np.polyfit(pico["bill_length_mm"], pico["bill_depth_mm"], 1)
xs = np.linspace(pico["bill_length_mm"].min(), pico["bill_length_mm"].max(), 50)
ejes[0].plot(xs, b0 + b1 * xs, color="crimson"); ejes[0].set_title(f"Todos juntos: r = {r_total:.2f}")
for e in especies:
    s = pico[pico["species"] == e]
    ejes[1].scatter(s["bill_length_mm"], s["bill_depth_mm"], s=12, alpha=0.6, color=colores[e], label=e)
    b1, b0 = np.polyfit(s["bill_length_mm"], s["bill_depth_mm"], 1)
    xe = np.linspace(s["bill_length_mm"].min(), s["bill_length_mm"].max(), 20)
    ejes[1].plot(xe, b0 + b1 * xe, color=colores[e])
ejes[1].set_title("Por especie"); ejes[1].legend()
for ax in ejes: ax.set_xlabel("longitud del pico (mm)")
ejes[0].set_ylabel("profundidad del pico (mm)")
plt.tight_layout(); plt.show()

# Modelo multiple: ¿qué signo tiene la pendiente al incluir la especie?
m_pico = smf.ols("bill_depth_mm ~ bill_length_mm + C(species)", data=pico).fit(cov_type="HC3")
print(f"Pendiente con la especie en el modelo: {m_pico.params['bill_length_mm']:+.3f} mm por mm (IC 95 % HC3: {m_pico.conf_int().loc['bill_length_mm', 0]:+.3f} a {m_pico.conf_int().loc['bill_length_mm', 1]:+.3f})")

### ¿Qué observas?

- En el total, la correlación es **negativa** (r ≈ −0,24): picos más largos parecen ir con picos menos profundos. Dentro de **cada especie** es **positiva** (≈ +0,39, +0,65 y +0,64).
- La causa es la composición: Gentoo tiene el pico más largo **pero** el menos profundo; Adelie, el más corto y profundo. Al mezclar, las diferencias **entre** especies dominan y dan la pendiente contraria a la que existe **dentro** de cada una.
- El modelo con la especie recupera la pendiente positiva. **Lección de proyecto:** antes de interpretar una asociación, pregúntate si hay una variable de grupo (segmento, canal, cohorte) que mezcla poblaciones. En el caso B del proyecto, la paradoja de Simpson y los segmentos son una trampa real.

---
## 7. Análisis de sensibilidad y límites

Un resultado es más creíble si **sobrevive** a decisiones razonables distintas. Comprobamos que la diferencia Gentoo − Adelie no depende de mezclar sexos y de los ausentes de `sex`.

In [ ]:
sens = []
for etiqueta, sub_ in [("Todos (n con masa)", masa),
                       ("Solo machos", masa[masa["sex"] == "Male"]),
                       ("Solo hembras", masa[masa["sex"] == "Female"])]:
    a = sub_.loc[sub_["species"] == "Adelie", "body_mass_g"]
    g = sub_.loc[sub_["species"] == "Gentoo", "body_mass_g"]
    t = stats.ttest_ind(g, a, equal_var=False)
    ic = t.confidence_interval(0.95)                  # IC de Welch para la diferencia de medias
    sens.append({"subconjunto": etiqueta, "n": len(sub_), "dif. Gentoo−Adelie (g)": round(g.mean() - a.mean()),
                 "IC 95 % inf.": round(ic.low), "IC 95 % sup.": round(ic.high)})
print(pd.DataFrame(sens).to_string(index=False))

### ¿Qué observas?

- La diferencia es grande (de ≈ 1 310 a ≈ 1 440 g) en los tres subconjuntos y los intervalos nunca se acercan a 0: **la conclusión es robusta** a esta decisión.
- Aun así, **límites** que el informe debe declarar: (1) los datos son **observacionales** y de **tres islas** de un archipiélago y tres años, no una muestra aleatoria de todos los pingüinos; (2) el supuesto de varianzas iguales solo se cumple de forma aproximada; (3) hay **11** filas con algún ausente que no participan en cada análisis; (4) una asociación entre masa y aleta no es una relación causal; (5) los pingüinos de una misma colonia no son estrictamente independientes.

---
## 8. El informe de una página

Un informe **no repite el notebook**: responde a la pregunta, con cifras, incertidumbre y límites, en una página. Plantilla: **Pregunta y decisión · Datos · Método · Resultados · Límites · Recomendación**. Lo escribimos en una celda que toma las cifras **de las variables**, para que no haya discrepancias entre el texto y el análisis.

In [ ]:
def num(x, d=0):
    return f"{x:.{d}f}".replace(".", ",")

def fila_tukey(g1, g2):
    return tabla_tukey[(tabla_tukey["group1"] == g1) & (tabla_tukey["group2"] == g2)].iloc[0]

ag, cg, ac = fila_tukey("Adelie", "Gentoo"), fila_tukey("Chinstrap", "Gentoo"), fila_tukey("Adelie", "Chinstrap")
n_masa = len(masa)

informe = f"""
### Informe: ¿distingue la masa corporal a las tres especies de pingüinos?

**Pregunta y decisión.** ¿Sirve una medida rápida de masa para identificar la especie? Decisión: si incluir la balanza en el protocolo de campo.

**Datos.** {len(pen)} pingüinos de Palmer (tres especies); {n_masa} con masa registrada (se descartan {len(pen) - n_masa} con ausentes). Sin duplicados ni valores imposibles. Datos observacionales.

**Método.** ANOVA de Welch (varianzas desiguales según Levene, p = {num(stats.levene(*grupos)[1], 3)}), confirmado con Kruskal-Wallis; comparaciones por pares con Tukey y bootstrap como segunda opinión.

**Resultados.**
- La masa difiere entre especies (Welch F = {welch.statistic:.0f}, p < 0,001; eta² = {num(eta2, 2)}: la especie explica el {eta2 * 100:.0f} % de la variación).
- Gentoo pesa {num(ag["meandiff"])} g más que Adelie (IC 95 %: {num(ag["lower"])} a {num(ag["upper"])}) y {num(cg["meandiff"])} g más que Chinstrap (IC 95 %: {num(cg["lower"])} a {num(cg["upper"])}).
- Adelie y Chinstrap **no** se distinguen por la masa ({num(ac["meandiff"])} g; IC 95 %: {num(ac["lower"])} a {num(ac["upper"])}, que incluye el 0).
- La diferencia Gentoo − Adelie se mantiene en machos y hembras por separado.

**Límites.** Datos observacionales de tres islas, no una muestra aleatoria de todos los pingüinos; varianzas solo aproximadamente iguales; observaciones de una misma colonia no estrictamente independientes. Los resultados describen asociación, no causa.

**Recomendación.** Usar la balanza para reconocer a **Gentoo**, pero no para separar Adelie de Chinstrap: para ese par hace falta otra medida (por ejemplo, la longitud del pico).
"""
display(Markdown(informe))

### ¿Qué observas?

- Cada afirmación lleva **efecto + IC**, no solo un p-value; los **límites** están escritos antes de la recomendación; y la recomendación responde a la **decisión** planteada al principio.
- El informe es corto porque decide qué **no** contar: los gráficos de supuestos, el bootstrap y los diagnósticos quedan en el notebook.

---
## 9. La rúbrica de diez criterios

Al terminar tu proyecto, puntúa cada criterio de 0 (no lo hice) a 2 (bien hecho). En la demostración:

| # | Criterio | ¿Dónde se ve aquí? |
|---|---|---|
| 1 | Pregunta y decisión definidas | Sección 1 |
| 2 | Auditoría de los datos | Sección 2 |
| 3 | Exploración antes de modelar | Sección 3 |
| 4 | Método justificado | Sección 4 (tabla de decisión) |
| 5 | Supuestos comprobados | Shapiro, Levene, QQ, residuos |
| 6 | Efecto + IC, no solo p-value | Tukey, bootstrap, HC3 |
| 7 | Multiplicidad o selección controladas | Tukey (3 pares a la vez) |
| 8 | Límites y sesgos declarados | Sección 7 |
| 9 | Sin afirmaciones causales indebidas | "asociación, no causa" |
| 10 | Comunicación clara y reproducible | Semilla fija, informe de una página |

## Resumen

- Un análisis va de la **pregunta** a la **decisión**; el código es el medio.
- **Audita** antes de calcular, **explora** antes de modelar, **justifica** el método y **comprueba** sus supuestos.
- Comunica **efecto + incertidumbre + límites**. Un p-value solo no es un resultado.
- Desconfía de las medias globales cuando hay grupos ocultos (Simpson) y de las asociaciones observacionales como causas.

## Ahora te toca

Abre `notebooks_practica/clase14_practica_proyecto_integrador.ipynb`, elige **uno** de los tres casos (A: salud, B: comercio electrónico con test A/B, C: viviendas) y llévalo hasta un informe de una página. Con esto **termina el curso**: para seguir, el paso natural es un curso de Machine Learning, que retoma la validación (clase 12) y los modelos de la clase 11 con herramientas de predicción.